In [31]:
# We need pandas for data manipulation, geopandas for spatial data,
# sqlalchemy to talk to PostgreSQL, shapely to create geometry points,
# os and dotenv to securely load our database credentials from .env file
import pandas as pd
import geopandas as gpd
from sqlalchemy import create_engine
from shapely.geometry import Point
import os
from dotenv import load_dotenv

In [10]:
# Load credentials from .env file so we never hardcode passwords in code
load_dotenv(r'D:\sfpris\.env')

DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')

# Create connection engine — this is our bridge between Python and PostgreSQL
engine = create_engine(f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')
print("DB connection ready")

DB connection ready


In [22]:
# PHMSA provides Excel files with two sheets each
# First sheet is the data dictionary (column descriptions)
# Second sheet is the actual incident data — we load that directly by name
hl = pd.read_excel(r'D:\sfpris\data\raw\hl2010toPresent.xlsx', sheet_name='hl2010toPresent')
gas = pd.read_excel(r'D:\sfpris\data\raw\gtggungs2010toPresent.xlsx', sheet_name='gtggungs2010toPresent')

print(f"Hazardous Liquid incidents: {hl.shape}")
print(f"Gas incidents: {gas.shape}")

Hazardous Liquid incidents: (5940, 668)
Gas incidents: (2018, 637)


In [23]:
# Both files have 600+ columns — we only need 13 relevant to our ML model
# We pick incident ID, year, cause, location, operator, cost, and pipe specs
# These will become our features for risk prediction
cols = [
    'REPORT_NUMBER',               # unique incident ID
    'IYEAR',                       # year incident occurred
    'SIGNIFICANT',                 # was it a significant incident YES/NO
    'CAUSE',                       # root cause of incident
    'LOCATION_LATITUDE',           # where it happened — latitude
    'LOCATION_LONGITUDE',          # where it happened — longitude
    'ONSHORE_STATE_ABBREVIATION',  # which US state
    'NAME',                        # pipeline operator name
    'TOTAL_COST',                  # total incident cost in USD
    'PIPE_DIAMETER',               # diameter of pipe in inches
    'MATERIAL_INVOLVED',           # pipe material eg carbon steel
    'INSTALLATION_YEAR',           # year pipe was installed
    'SYSTEM_PART_INVOLVED'         # which part of the system failed
]

hl_clean = hl[cols].copy()
gas_clean = gas[cols].copy()

print(f"HL clean shape: {hl_clean.shape}")
print(f"Gas clean shape: {gas_clean.shape}")

HL clean shape: (5940, 13)
Gas clean shape: (2018, 13)


In [26]:
from pandas import isnull


hl_clean(['LOCATION_LATITUDE']).isnull.sum()

TypeError: 'DataFrame' object is not callable

In [24]:
# Rows without latitude or longitude cannot be mapped or used in spatial analysis
# We drop them here — they are useless for our geospatial ML project
# reset_index keeps the index clean after dropping rows
hl_clean = hl_clean.dropna(subset=['LOCATION_LATITUDE', 'LOCATION_LONGITUDE']).reset_index(drop=True)
gas_clean = gas_clean.dropna(subset=['LOCATION_LATITUDE', 'LOCATION_LONGITUDE']).reset_index(drop=True)

print(f"HL after dropping null coordinates: {hl_clean.shape}")
print(f"Gas after dropping null coordinates: {gas_clean.shape}")

HL after dropping null coordinates: (5940, 13)
Gas after dropping null coordinates: (2018, 13)


In [27]:
# We add a source column to track which dataset each record came from
# This helps when we combine or query both tables together later
hl_clean['source'] = 'HAZARDOUS_LIQUID'
gas_clean['source'] = 'GAS_TRANSMISSION'

print("Source columns added.")

Source columns added.


In [28]:
# This is the most important spatial step in this notebook
# We convert lat/lon columns into actual geometry Point objects using Shapely
# Then wrap the dataframe in a GeoDataFrame so it understands spatial operations
# EPSG 4326 = WGS84 = standard global lat/lon coordinate system
hl_gdf = gpd.GeoDataFrame(
    hl_clean,
    geometry=gpd.points_from_xy(hl_clean.LOCATION_LONGITUDE, hl_clean.LOCATION_LATITUDE),
    crs='EPSG:4326'
)

gas_gdf = gpd.GeoDataFrame(
    gas_clean,
    geometry=gpd.points_from_xy(gas_clean.LOCATION_LONGITUDE, gas_clean.LOCATION_LATITUDE),
    crs='EPSG:4326'
)

print(f"HL GeoDataFrame: {hl_gdf.shape}")
print(f"Gas GeoDataFrame: {gas_gdf.shape}")
print(hl_gdf.geometry.head(3))

HL GeoDataFrame: (5940, 15)
Gas GeoDataFrame: (2018, 15)
0     POINT (-88.23353 41.94352)
1    POINT (-100.80037 37.10847)
2     POINT (-101.4044 32.22471)
Name: geometry, dtype: geometry


In [45]:
# Push both GeoDataFrames into our PostGIS tables
# to_postgis handles geometry conversion automatically
# if_exists='append' adds to existing table without dropping it
# Our tables were already created in pgAdmin with the correct schema
print("Loading Hazardous Liquid incidents into PostGIS...")
hl_gdf.to_postgis(
    name='phmsa_hl_incidents',
    con=engine,
    if_exists='append',
    index=False
)
print(f"HL loaded: {len(hl_gdf)} rows")

print("Loading Gas incidents into PostGIS...")
gas_gdf.to_postgis(
    name='phmsa_gas_incidents',
    con=engine,
    if_exists='append',
    index=False
)
print(f"Gas loaded: {len(gas_gdf)} rows")

print("PHMSA data loading complete.")

Loading Hazardous Liquid incidents into PostGIS...
HL loaded: 5940 rows
Loading Gas incidents into PostGIS...
Gas loaded: 2018 rows
PHMSA data loading complete.
